# navidet 사용 예시

한 패키지에서 두 계열 모델을 코드로 직접 쓰는 방법을 보여준다.

- **A. 모델 생성 + forward** (`build_model`) — 학습된 가중치 없이도 동작
- **B. 6DoF 추론** (`YOLO6DoFPredictor`) — 체크포인트 있으면 실행
- **C. Pose 추론** (`YOLOPose` 직접 디코딩) — 박스 + 키포인트 시각화

> 체크포인트/데이터가 없어도 노트북 전체가 끝까지 실행되도록 랜덤 가중치 폴백을 둔다.
> 실제 가중치가 있으면 아래 `CKPT_6DOF` / `CKPT_POSE` 경로만 채우면 된다.

## 0. 셋업

In [ ]:
import os, sys
from pathlib import Path

# repo 루트(navidet 패키지가 있는 곳)를 import 경로에 추가
ROOT = Path.cwd()
if not (ROOT / 'navidet').is_dir() and (ROOT.parent / 'navidet').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import navidet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('navidet OK | device =', device)
print('exports:', [n for n in navidet.__all__ if 'YOLO' in n or 'Loss' in n])

# 학습된 체크포인트가 있으면 경로를 채운다 (없으면 랜덤 가중치로 동작)
CKPT_6DOF = 'runs/exp_distill/best.pt'
CKPT_POSE = 'runs/exp_s/best.pt'

## A. 모델 생성 + forward (`build_model`)

`task` 로 계열을 고른다: `"6dof" | "detect" | "pose"` (+`segment`).

In [ ]:
from navidet import build_model

# 6DoF 모델
m6 = build_model('6dof', nc=13, scale='n').to(device).eval()
# Pose 모델 (키포인트 4개, 차원 3=xy+visibility)
mp = build_model('pose', nc=3, scale='n', kpt_shape=(4, 3)).to(device).eval()

x = torch.randn(1, 3, 320, 320, device=device)
with torch.no_grad():
    o6 = m6(x)   # eval: {'det', 'depth'}
    op = mp(x)   # eval: {'boxes', 'scores', 'kpt'}

n_params = lambda m: sum(p.numel() for p in m.parameters()) / 1e6
print(f'YOLO6DoF  params={n_params(m6):.2f}M  det={tuple(o6["det"].shape)}  depth={tuple(o6["depth"].shape)}')
print(f'YOLOPose  params={n_params(mp):.2f}M  boxes={tuple(op["boxes"].shape)}  '
      f'scores={tuple(op["scores"].shape)}  kpt={tuple(op["kpt"].shape)}')

### (참고) 학습 1-step — 손실 + backward
데이터 없이 합성 타깃으로 손실/역전파가 도는지만 확인한다.

In [ ]:
from navidet.module.loss_mt import MultiTaskLoss

B, nk, D = 2, 4, 3
model = build_model('pose', nc=3, scale='n', kpt_shape=(nk, D)).to(device).train()
loss_fn = MultiTaskLoss(nc=3, strides=model.head.strides, tasks=('detect', 'pose'),
                        kpt_shape=(nk, D)).to(device)

out = model(torch.randn(B, 3, 320, 320, device=device))   # train: raw dict
targets = {
    'gt_labels': torch.tensor([[[0.], [1.]]] * B, device=device),
    'gt_bboxes': torch.tensor([[[20., 20., 60., 60.], [80., 80., 120., 120.]]] * B, device=device),
    'mask_gt':   torch.ones(B, 2, 1, device=device),
    'gt_kpts':   torch.rand(B, 2, nk, D, device=device) * 100,
}
targets['gt_kpts'][..., 2] = 1.0   # visibility=가시

loss, items = loss_fn(out, targets)
loss.backward()
print('loss =', round(float(loss), 3), '| finite =', bool(torch.isfinite(loss)))
print({k: round(v, 3) for k, v in items.items()})

## B. 6DoF 추론 (`YOLO6DoFPredictor`)

체크포인트 + (선택)카메라 intrinsics만 주면 BGR 프레임 하나로 `R/t/size`를 얻는다.
체크포인트가 없으면 이 셀은 건너뛴다.

In [ ]:
from navidet.module.predictor import YOLO6DoFPredictor

ckpt = ROOT / CKPT_6DOF
if ckpt.exists():
    predictor = YOLO6DoFPredictor(str(ckpt), ini=None, conf=0.25, iou=0.5)
    # 데모용 더미 BGR 프레임 (실제로는 cv2.imread('test.png'))
    frame = (np.random.rand(720, 1280, 3) * 255).astype(np.uint8)
    dets = predictor(frame)
    print(f'검출 {len(dets)}개')
    for d in dets[:3]:
        print(f"  cls={d['cls']} conf={d['conf']:.2f} t={np.round(d['t'], 3)} size={np.round(d['size'], 3)}")
else:
    print(f'[skip] 체크포인트 없음: {ckpt}  (CKPT_6DOF 경로를 채우면 실행됩니다)')

## C. Pose 추론 (`YOLOPose` 직접 디코딩 + 시각화)

전처리(letterbox) → forward → 박스/키포인트 디코딩 → NMS → 그리기.
학습된 가중치가 있으면 로드하고, 없으면 랜덤 가중치로도 파이프라인 전체가 실행된다
(결과는 무의미하지만 동작 확인용).

In [ ]:
from PIL import Image, ImageDraw
from navidet.core.model import YOLOPose, TASK_PRESET
from navidet.module.dataset import letterbox_rgb
from navidet.utils.nms import nms, xywh2xyxy

# --- 모델 구성: 체크포인트 있으면 로드, 없으면 랜덤 ---
ckpt = ROOT / CKPT_POSE
if ckpt.exists():
    ck = torch.load(str(ckpt), map_location=device, weights_only=False)
    task = ck.get('task', 'pose'); kpt_shape = tuple(ck.get('kpt_shape', (4, 3)))
    nc, scale, imgsz = ck['nc'], ck['scale'], ck['imgsz']
    pose_model = YOLOPose(nc=nc, scale=scale, tasks=TASK_PRESET[task],
                          nm=ck.get('nm', 32), kpt_shape=kpt_shape).to(device).eval()
    pose_model.load_state_dict(ck['model'])
    print('체크포인트 로드:', ckpt)
else:
    nc, scale, imgsz, kpt_shape = 3, 'n', 640, (4, 3)
    pose_model = build_model('pose', nc=nc, scale=scale, kpt_shape=kpt_shape).to(device).eval()
    print('[랜덤 가중치] 체크포인트 없음 → 파이프라인 동작만 시연')

# --- 입력 이미지: 실제 이미지가 있으면 사용, 없으면 합성 ---
IMG_PATH = ROOT / 'test.png'
if IMG_PATH.exists():
    img = Image.open(IMG_PATH).convert('RGB')
else:
    arr = (np.random.rand(480, 640, 3) * 255).astype(np.uint8)
    img = Image.fromarray(arr)
    print('[합성 이미지 사용]')

In [ ]:
# --- 전처리(letterbox) → forward → 디코딩 ---
conf_th, iou_th = 0.25, 0.5
rgb, r, left, top = letterbox_rgb(img, imgsz)
x = torch.from_numpy(rgb).permute(2, 0, 1).float().div(255)[None].to(device)

with torch.no_grad():
    dec = pose_model(x)
boxes  = dec['boxes'][0].T                         # [A,4] xywh(px, letterbox)
scores = dec['scores'][0]                          # [nc,A]
nk, Dk = kpt_shape
kpt    = dec['kpt'][0].view(nk, Dk, -1).permute(2, 0, 1)   # [A,nk,Dk]

cls_score, cls_id = scores.max(0)
idx = (cls_score > conf_th).nonzero().squeeze(1)
sel = idx[torch.tensor(nms(xywh2xyxy(boxes[idx]), cls_score[idx], iou_th))] if idx.numel() else idx
print(f'conf>{conf_th} 후보 {idx.numel()}개 → NMS 후 {len(sel)}개')

# 원본 좌표로 역변환 헬퍼: (letterbox px) → (원본 px)
to_orig = lambda px, py: ((px - left) / r, (py - top) / r)
for i in sel.tolist()[:5]:
    x1, y1, x2, y2 = xywh2xyxy(boxes[i]).tolist()
    ox1, oy1 = to_orig(x1, y1); ox2, oy2 = to_orig(x2, y2)
    print(f"  cls={int(cls_id[i])} conf={float(cls_score[i]):.2f} "
          f"box_orig=({ox1:.0f},{oy1:.0f},{ox2:.0f},{oy2:.0f})")

In [ ]:
# --- 시각화 (letterbox 이미지 위에 박스 + 키포인트) ---
import matplotlib.pyplot as plt

vis = Image.fromarray(rgb.copy()); draw = ImageDraw.Draw(vis)
palette = [(0, 255, 255), (255, 128, 0), (255, 0, 255), (0, 255, 0)]
for i in sel.tolist():
    c = palette[int(cls_id[i]) % len(palette)]
    x1, y1, x2, y2 = xywh2xyxy(boxes[i]).tolist()
    draw.rectangle([x1, y1, x2, y2], outline=c, width=2)
    for k in range(nk):
        kx, ky = kpt[i, k, 0].item(), kpt[i, k, 1].item()
        vbl = kpt[i, k, 2].item() if Dk == 3 else 1.0
        if vbl > 0.3:
            draw.ellipse([kx - 3, ky - 3, kx + 3, ky + 3], fill=c)

plt.figure(figsize=(7, 7)); plt.imshow(vis); plt.axis('off')
plt.title(f'pose: {len(sel)} det (conf>{conf_th})'); plt.show()

---
### 정리

| 용도 | 쓰는 것 |
|---|---|
| 6DoF 실시간 추론 (간편) | `YOLO6DoFPredictor` |
| 모델 생성 / 학습·파인튜닝 | `build_model(task=...)` |
| Pose 추론 (박스+키포인트) | `YOLOPose` 직접 로드 + 디코딩 |

학습은 CLI로: `python -m navidet.tools.train --config navidet/config/default_pose.yaml`